# 📡 Advanced Forecasting
**fpppy Chapters 10–12 · Data Pattern Recognition for the Rest of Us**

> Prophet handles complex business seasonality, promotions, and holidays automatically. ML-based forecasting with lag features scales to thousands of SKUs. This notebook shows the modern retail forecasting toolkit.

### Required reading
📘 [fpppy Ch 10–12](https://otexts.com/fpppy/)

### Dataset
**Retail Store Sales** — weekly sales for a multi-department retailer with promotional events, holidays, and seasonal patterns. Modelled on M5 competition and Walmart sales data characteristics.

### Time: ~70 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
!pip install prophet -q
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
print("\u2713 Setup complete — Prophet installed")

In [ ]:
# Retail Store Weekly Sales — realistic M5-style dataset
# Features: trend, weekly/annual seasonality, holiday spikes, promotions
np.random.seed(42)

dates = pd.date_range('2018-01-01', '2023-12-31', freq='W-MON')
n = len(dates)

# Trend: slow growth
trend = 50000 + np.linspace(0, 20000, n)

# Annual seasonality: back-to-school (Aug), holiday (Nov-Dec), post-holiday drop (Jan)
doy = np.array([d.timetuple().tm_yday for d in dates])
annual = (8000 * np.sin(2*np.pi*(doy-80)/365)
         + 4000 * np.sin(4*np.pi*(doy-40)/365))

# Weekly seasonality already in weekly data (aggregated)

# Holiday spikes
holiday_boost = np.zeros(n)
for i, d in enumerate(dates):
    if d.month == 11 and d.day >= 20:   # Thanksgiving week
        holiday_boost[i] = 25000
    elif d.month == 12 and d.day <= 27: # Christmas
        holiday_boost[i] = 35000
    elif d.month == 7 and 1<=d.day<=7:  # July 4th
        holiday_boost[i] = 8000
    elif d.month == 1 and d.day <= 7:   # Post-holiday drop
        holiday_boost[i] = -15000

# Promotions (random but correlated with slow periods)
promotion = np.random.choice([0, 5000, 12000, 20000], n, p=[0.6, 0.2, 0.12, 0.08])

# Noise
noise = np.random.normal(0, 3500, n)

# Final sales
sales = (trend + annual + holiday_boost + promotion + noise).clip(10000)

retail = pd.DataFrame({'ds': dates, 'y': sales.round(-2),
                        'promotion': (promotion > 0).astype(int)})

print(f"Retail Sales dataset: {len(retail)} weekly observations")
print(f"Date range: {retail['ds'].min().date()} to {retail['ds'].max().date()}")
print(f"Weekly sales range: ${retail['y'].min():,.0f} — ${retail['y'].max():,.0f}")
print(f"Median weekly sales: ${retail['y'].median():,.0f}")
print(f"Promotion weeks: {retail['promotion'].sum()} ({retail['promotion'].mean():.1%})")

In [ ]:
# Exploratory: visualize the components we built in
fig, axes = plt.subplots(3, 1, figsize=(13, 9))

axes[0].plot(retail['ds'], retail['y']/1000, color='#1e5fa8', lw=1.2, alpha=0.8)
axes[0].set_title('Weekly Retail Sales — Multi-Year View')
axes[0].set_ylabel('Sales ($000s)')

# Zoom into one year
one_year = retail[retail['ds'].dt.year == 2022]
axes[1].plot(one_year['ds'], one_year['y']/1000, color='#e85d20', lw=1.5)
promo_weeks = one_year[one_year['promotion']==1]
axes[1].scatter(promo_weeks['ds'], promo_weeks['y']/1000,
                color='#1a7a45', s=60, zorder=3, label='Promotion week')
axes[1].set_title('2022 Weekly Sales — Seasonal Pattern + Promotions')
axes[1].set_ylabel('Sales ($000s)')
axes[1].legend()

# Year-over-year comparison
for year in [2020, 2021, 2022, 2023]:
    yr_data = retail[retail['ds'].dt.year == year]
    if len(yr_data) >= 50:
        axes[2].plot(range(len(yr_data)), yr_data['y']/1000,
                    lw=1.5, alpha=0.8, label=str(year))
axes[2].set_title('Year-over-Year Comparison')
axes[2].set_xlabel('Week of Year')
axes[2].set_ylabel('Sales ($000s)')
axes[2].legend()

plt.tight_layout(); plt.show()

## 🔮 Part 1 — Prophet: Business Forecasting with Seasonality & Events

Prophet decomposes sales as:
```
y(t) = trend(t) + weekly_seasonality(t) + annual_seasonality(t) + holidays(t) + ε
```

The holiday/promotion component is critical for retail — holiday weeks are 40-60% above baseline, and models that ignore them produce terrible forecasts.

In [ ]:
# Temporal train/test split — never shuffle time series
split_date = '2023-01-01'
train = retail[retail['ds'] < split_date][['ds','y']]
test  = retail[retail['ds'] >= split_date][['ds','y']]
h = len(test)

print(f"Training: {len(train)} weeks ({train['ds'].min().date()} to {train['ds'].max().date()})")
print(f"Test:     {h} weeks ({test['ds'].min().date()} to {test['ds'].max().date()})")

# Define retail holidays
from prophet.make_holidays import make_holidays_df
# Manual holiday list for retail
holidays = pd.DataFrame({
    'holiday': ['thanksgiving','christmas','july4','postthanksgiving','post_holiday'],
    'ds': pd.to_datetime([
        '2018-11-22','2019-11-28','2020-11-26','2021-11-25','2022-11-24',
        '2018-12-24','2019-12-23','2020-12-21','2021-12-20','2022-12-19',
        '2018-07-02','2019-07-01','2020-06-29','2021-06-28','2022-07-04',
        '2018-11-26','2019-12-02','2020-11-30','2021-11-29','2022-11-28',
        '2019-01-07','2020-01-06','2021-01-04','2022-01-03','2023-01-02',
    ]),
    'lower_window': [0,0,0,0,-2],
    'upper_window': [1,1,1,2,0]
})

# Fit Prophet
m = Prophet(yearly_seasonality=True, weekly_seasonality=False,
            holidays=holidays,
            changepoint_prior_scale=0.08,
            seasonality_prior_scale=15,
            holidays_prior_scale=20)
m.fit(train)

future = m.make_future_dataframe(periods=h, freq='W')
forecast = m.predict(future)
test_forecast = forecast[forecast['ds'].isin(test['ds'])]

mae_prophet = mean_absolute_error(test['y'], test_forecast['yhat'])
mape_prophet= (abs(test['y'].values - test_forecast['yhat'].values) / test['y'].values).mean()

print(f"\nProphet test performance:")
print(f"  MAE:  ${mae_prophet:,.0f}")
print(f"  MAPE: {mape_prophet:.1%}")

In [ ]:
# Prophet forecast plot
fig = m.plot(forecast, figsize=(13, 5))
fig.axes[0].set_title('Prophet Forecast — Retail Weekly Sales')
fig.axes[0].axvline(pd.Timestamp(split_date), color='#e85d20', lw=2,
                    ls='--', label='Train/Test split')
fig.axes[0].legend()
plt.tight_layout(); plt.show()

# Components
fig2 = m.plot_components(forecast, figsize=(12, 9))
fig2.suptitle('Prophet Components: Trend + Seasonality + Holidays', y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# ML approach: GBM with lag features
def make_lag_features(df, target='y', lags=[1,2,4,8,13,26,52]):
    df = df.copy().set_index('ds')
    for lag in lags:
        df[f'lag_{lag}w'] = df[target].shift(lag)
    df['rolling_4w_mean'] = df[target].shift(1).rolling(4).mean()
    df['rolling_13w_mean']= df[target].shift(1).rolling(13).mean()
    df['week_of_year']    = df.index.isocalendar().week.astype(int)
    df['month']           = df.index.month
    return df.dropna()

retail_feat = make_lag_features(retail.drop('promotion',axis=1))
feature_cols= [c for c in retail_feat.columns if c != 'y']

split_idx = retail_feat.index < pd.Timestamp(split_date)
X_tr_ml = retail_feat.loc[split_idx, feature_cols]
y_tr_ml = retail_feat.loc[split_idx, 'y']
X_te_ml = retail_feat.loc[~split_idx, feature_cols]
y_te_ml = retail_feat.loc[~split_idx, 'y']

gbm_ts = GradientBoostingRegressor(n_estimators=300, max_depth=4,
                                    learning_rate=0.05, random_state=42)
gbm_ts.fit(X_tr_ml, y_tr_ml)
y_pred_gbm = gbm_ts.predict(X_te_ml)

mae_gbm  = mean_absolute_error(y_te_ml, y_pred_gbm)
mape_gbm = (abs(y_te_ml.values - y_pred_gbm) / y_te_ml.values).mean()

# Naive baseline (same week last year)
naive_pred = retail_feat.loc[~split_idx, 'lag_52w'].values
mae_naive  = mean_absolute_error(y_te_ml, naive_pred)
mape_naive = (abs(y_te_ml.values - naive_pred) / y_te_ml.values).mean()

print("=== Model Comparison — Retail Sales Forecast ===\n")
print(f"{'Model':<25} {'MAE':>10} {'MAPE':>8}")
print("-"*45)
for name, mae, mape in [
    ('Naive (same wk last yr)', mae_naive,   mape_naive),
    ('Prophet',                 mae_prophet, mape_prophet),
    ('GBM + lag features',      mae_gbm,     mape_gbm)]:
    print(f"  {name:<23} ${mae:>9,.0f} {mape:>7.1%}")

# Feature importance for GBM
imp = pd.Series(gbm_ts.feature_importances_, index=feature_cols).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(9,4))
ax.bar(range(len(imp)), imp.values, color='#1e5fa8', edgecolor='white')
ax.set_xticks(range(len(imp)))
ax.set_xticklabels(imp.index, rotation=45, ha='right', fontsize=9)
ax.set_title('GBM Feature Importance — Retail Sales\n'
             'Lag features + seasonality drive predictions')
plt.tight_layout(); plt.show()

In [ ]:
answers = {
    "q1": "",  # What is the key difference between Prophet and ARIMA?
    "q2": "",  # Why can you NOT randomly shuffle train/test split for time series?
    "q3": "",  # How do you convert a time series into a supervised ML problem?
    "q4": "",  # What does changepoint_prior_scale control in Prophet?
    "q5": "",  # For retail sales, when would you choose GBM+lags over Prophet?
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print("\u274c Still empty: "+str(missing) if missing else "\u2705 Done! File \u2192 Save a copy in GitHub")

### 📤 Submit for AI Grading

In [ ]:
_NB_NAME_ = "Advanced Forecasting"
# ══════════════════════════════════════════════════════════════════════
# 🤖 AI-GRADED SUBMISSION
# ══════════════════════════════════════════════════════════════════════
#
# REQUIRED — fill in your GitHub username so your instructor can
# match your submission to your name:
#
GITHUB_USERNAME = ""   # ← e.g. "jsmith42"  — your github.com username
#
# ── ONE-TIME SETUP (do this once, works for all 30 notebooks) ─────────
#
# You need a FREE Gemini API key from Google AI Studio.
#
# ⚠️  IMPORTANT: Use your PERSONAL Gmail — NOT your university email.
#     Many universities block Google AI Studio on managed accounts.
#     A personal @gmail.com works everywhere and is always free.
#
# Steps:
#   1. Open a private/incognito browser window
#   2. Go to: aistudio.google.com
#   3. Sign in with your PERSONAL Gmail (@gmail.com)
#   4. Click "Get API key" → "Create API key" → copy it
#   5. Back in Colab: click the 🔑 icon in the LEFT SIDEBAR
#      → "+ Add new secret"
#      → Name:  GEMINI_API_KEY
#      → Value: paste your key here
#      → Toggle the switch ON
#   6. Re-run this cell
#
# Your key is saved to your Colab account — works across all notebooks.
# It is FREE — no credit card, no billing required.
#
# ── NO KEY? ────────────────────────────────────────────────────────────
# You still get completion-based feedback without a key.
# You just won't get AI analysis of your specific answers.
#
# ══════════════════════════════════════════════════════════════════════
# DO NOT EDIT BELOW THIS LINE
# ══════════════════════════════════════════════════════════════════════
import os, json, textwrap, urllib.request, re as _re

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _get_gemini_key():
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            return key
    except Exception:
        pass
    return None

def _call_gemini(prompt, api_key):
    url = (f"https://generativelanguage.googleapis.com/v1beta/"
           f"models/gemini-2.0-flash:generateContent?key={api_key}")
    payload = json.dumps({
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"maxOutputTokens": 1024, "temperature": 0.3}
    }).encode()
    req = urllib.request.Request(
        url, data=payload,
        headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read())
            return data["candidates"][0]["content"]["parts"][0]["text"]
    except urllib.error.HTTPError as e:
        return f"API_ERROR:{e.code}:{e.read().decode()[:200]}"
    except Exception as e:
        return f"ERROR:{e}"

def _build_prompt(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    n_done   = len(answered)
    qa_lines = "\n".join(
        f"Q{k.replace('q','')}: {v}" for k, v in answered.items()
    )
    return f"""You are a helpful TA grading quiz answers for the \"{nb_name}\" notebook
in a data science course called \"Data Pattern Recognition for the Rest of Us\",
based on ISLP (Introduction to Statistical Learning with Python).

## Student Answers ({n_done}/{n_total} questions answered)
{qa_lines if qa_lines else "(none provided)"}

## Instructions
- Grade conceptual understanding, not exact wording
- Accept any reasonable paraphrase of the correct answer
- Be encouraging, specific, and concise
- Respond ONLY with valid JSON, no markdown fences, no extra text:

{{
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent | Good | Needs Review | Incomplete>",
  "feedback": "<2-3 sentences: what they got right, any misconceptions to fix>",
  "tip": "<one specific thing to remember or explore from this technique>"
}}"""

def _rule_based_grade(answers_dict):
    n_total    = len(answers_dict)
    answered   = [v for v in answers_dict.values() if str(v).strip()]
    n_answered = len(answered)
    avg_len    = sum(len(str(v)) for v in answered) / max(n_answered, 1)
    score      = n_answered
    if n_answered == 0:
        return {"quiz_score": 0, "grade": "Incomplete",
                "feedback": "No answers provided. Fill in the quiz answers above and re-run.",
                "tip": "Each question only needs 1-2 sentences."}
    elif avg_len < 15:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"You answered {n_answered}/{n_total} questions but most "
                             "responses are very brief. Add more detail to show your understanding."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}
    elif n_answered == n_total:
        return {"quiz_score": score, "grade": "Good",
                "feedback": (f"All {n_total} questions answered with reasonable detail. "
                             "Add a free Gemini key for AI analysis of your specific answers."),
                "tip": "Get a free key at aistudio.google.com — use your personal Gmail."}
    else:
        return {"quiz_score": score, "grade": "Needs Review",
                "feedback": (f"{n_answered} of {n_total} questions answered. "
                             "Complete the remaining questions for full credit."),
                "tip": "Add your Gemini API key (personal Gmail) for detailed AI feedback."}

def _print_result(result, username, nb_name):
    colors = {
        "Excellent":    "\033[92m",
        "Good":         "\033[94m",
        "Needs Review": "\033[93m",
        "Incomplete":   "\033[91m",
    }
    R     = "\033[0m"
    grade = result.get("grade", "See feedback")
    C     = colors.get(grade, "\033[0m")
    n     = len(answers)
    qs    = result.get("quiz_score", 0)
    bar   = "\u2588" * qs + "\u2591" * (n - qs)

    print("\n" + "\u2550"*58)
    print(f"  \U0001f916  AI Feedback \u2014 {nb_name}")
    print("\u2550"*58)
    if username:
        print(f"  Student : @{username}")
    else:
        print(f"  Student : \u26a0\ufe0f  Set GITHUB_USERNAME above to track submissions")
    print(f"  Grade   : {C}{grade}{R}")
    print(f"  Score   : {qs}/{n}  [{bar}]")
    print()
    fb = result.get("feedback", "")
    for line in textwrap.wrap(fb, width=56):
        print(f"  {line}")
    tip = result.get("tip", "")
    if tip:
        print()
        for line in textwrap.wrap(f"\U0001f4a1 {tip}", width=56):
            print(f"  {line}")
    print("\u2550"*58 + "\n")

# ── Main grading flow ─────────────────────────────────────────
print("\n" + "\u2500"*58)
print("  Checking submission...")
print("\u2500"*58)

if "answers" not in globals():
    print("  \u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    n_total    = len(answers)
    n_answered = sum(1 for v in answers.values() if str(v).strip())
    username   = GITHUB_USERNAME.strip()

    print(f"  Notebook : {_NB_TITLE}")
    print(f"  Answers  : {n_answered}/{n_total} provided")
    if username:
        print(f"  Student  : @{username}")
    else:
        print(f"  Student  : \u26a0\ufe0f  GITHUB_USERNAME not set")

    api_key = _get_gemini_key()

    if api_key:
        print(f"  AI model : Gemini 2.0 Flash \u2713")
        print(f"  Grading  : please wait 10-20 seconds...")
        raw = _call_gemini(_build_prompt(answers, _NB_TITLE), api_key)
        if raw.startswith("API_ERROR") or raw.startswith("ERROR"):
            print(f"  \u26a0  {raw[:120]}")
            result = _rule_based_grade(answers)
        else:
            try:
                clean  = _re.sub(r"```(?:json)?\s*|```", "", raw).strip()
                result = json.loads(clean)
            except Exception:
                result = {"quiz_score": n_answered,
                          "grade": "Good" if n_answered == n_total else "Needs Review",
                          "feedback": raw[:400], "tip": ""}
    else:
        print("  AI model : rule-based (no Gemini key found)")
        print()
        print("  To enable AI grading:")
        print("  1. aistudio.google.com \u2192 Get API key (free, personal Gmail)")
        print("  2. Colab left sidebar \u2192 \U0001f511 Secrets")
        print("     Name: GEMINI_API_KEY  |  Value: your key  |  Toggle: ON")
        print("  3. Re-run this cell")
        print()
        result = _rule_based_grade(answers)

    _print_result(result, username, _NB_TITLE)
    print("  \U0001f4be Save: File \u2192 Save a copy in GitHub \u2192 your fork")
    print()


---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*